### RAG(Retrieval-Augmented Generation), 검색 증강 생성
- 생성형 AI가 학습하지 않은 최신 정보나 특정 조직의 내부 데이터를 바탕으로 답변할 수 있도록 돕는 기술이다.
- R (Retrieval): 질문과 관련된 문서를 방대한 데이터베이스에서 검색한다.
- A (Augmentation): 검색된 정보를 질문과 결합하여 프롬프트를 보강한다.
- G (Generation): 보강된 맥락을 바탕으로 최종 답변을 생성한다.

### RAG 개발 순서
1. 문서 불러오기
2. 문서 분할하기
3. 임베딩
4. Vector DB 생성 및 저장
5. 검색기 생성
6. 프롬프트 생성
7. LLM 생성
8. 체인 실행

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

# 1단계, 문서 로드
FILE_PATH = "./documents/ai커리큘럼.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()

print(f"문서의 페이지 수: {len(docs)}")

In [ ]:
print(docs[0].page_content)

In [ ]:
docs[0].__dict__

In [ ]:
# 2단계, 문서분할

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크의 수: {len(split_documents)}")

In [ ]:
# 3단계, 임베딩
embeddings = OpenAIEmbeddings()

In [ ]:
# 4단계, 벡터 DB

vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

In [ ]:
vectorstore.similarity_search("PyTorch")[1].page_content

In [ ]:
# 5단계, 검색기 생성

retriever = vectorstore.as_retriever(search_kwargs={"k":1})

retriever.invoke("프로젝트")

In [ ]:
# 6단계, 프롬프트 생성
prompt_template = PromptTemplate.from_template(
    """
    귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

    #Context: 
    {context}
    
    #Question:
    {question}
    
    #Answer:
    """
)

In [ ]:
# 7단계, LLM 생성

llm = ChatOpenAI(
    model_name = "gpt-4.1-nano",
    temperature=0
)

In [ ]:
# 8단계, 체인 생성

chain = (
    {"context" : retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [ ]:
question = "LangChain 체인설계는 몇 주차에 배우는가?"
answer = chain.invoke(question)
print(answer)